In [8]:
import os
import geopandas as gpd
import pandas as pd
import osmnx as ox


OUTDIR = "./data/green_jundiai"
os.makedirs(OUTDIR, exist_ok=True)

# 1) Limite municipal de Jundiaí
place = "Jundiaí, São Paulo, Brasil"
muni = ox.geocode_to_gdf(place)            # boundary em WGS84 (EPSG:4326)

# 2) Parques e jardins (leisure=park|garden, + áreas de recreação)
tags_parques = {"leisure": ["park", "garden"], "landuse": ["recreation_ground"]}
parks  = ox.features_from_place(place, tags_parques)

# 3) Praças
# - Em OSM, “praça” aparece principalmente como:
#   a) place=square (ponto ou polígono)
#   b) áreas com nome iniciando por “Praça ...” (mesmo que tag não seja place=square)
tags_pracas = {"place": "square"}
squares  = ox.features_from_place(place, tags_pracas)


# 3b) Capturar polígonos com nome "Praça ..." mesmo sem place=square
#     (ex.: áreas pedestrian, highway=pedestrian area, etc.)
tags_poligonos_diversos = {"highway": "pedestrian", "area": "yes"}
ped_areas = ox.features_from_place(place, tags_poligonos_diversos)
ped_areas = ped_areas[ped_areas.get("name", "").astype(str).str.startswith("Praça")]

# 4) Padronizar CRS e recortar pelo limite (só para garantir)
def clean_clip(gdf):
    if gdf.empty:
        return gdf
    gdf = gdf.set_crs(4326, allow_override=True)
    gdf = gpd.clip(gdf, muni.to_crs(4326))
    # manter só geometrias de área p/ shapefile de polígonos
    return gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]

parks_poly  = clean_clip(parks)
squares_poly = clean_clip(squares)
ped_pracas_poly = clean_clip(ped_areas)

# 5) Unir as praças de múltiplas fontes
pracas_poly = gpd.GeoDataFrame(pd.concat([squares_poly, ped_pracas_poly], ignore_index=True), crs=4326)

# 6) Salvar Shapefiles
parks_poly.to_file(os.path.join(OUTDIR, "parques_jundiai.shp"))
pracas_poly.to_file(os.path.join(OUTDIR, "pracas_jundiai.shp"))
muni.to_file(os.path.join(OUTDIR, "limite_municipal_jundiai.shp"))

print("✅ Salvo em:", OUTDIR)

✅ Salvo em: ./data/green_jundiai


d:\Users\ivan.cavalcanti\AppData\Local\Temp\16\ipykernel_119248\3241957715.py:49: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  parks_poly.to_file(os.path.join(OUTDIR, "parques_jundiai.shp"))
d:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\.venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'garden:type' to 'garden_typ'
  ogr_write(
d:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\.venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'addr:city' to 'addr_city'
  ogr_write(
d:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\.venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'addr:housenumber' to 'addr_house'
  ogr_write(
d:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\.venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'addr:postcode' to 'a

In [66]:
import geopandas as gpd
import folium


# reproject to WGS84 (lat/lon) if needed
if parques.crs and parques.crs.to_string() != "EPSG:4326":
   parques = parques.to_crs(epsg=4326)

   # reproject to WGS84 (lat/lon) if needed
if pracas.crs and pracas.crs.to_string() != "EPSG:4326":
   pracas = pracas.to_crs(epsg=4326)

# --- base map ---
m = folium.Map(location=[-15, -55], zoom_start=5, tiles="CartoDB positron")
# --- add shapefile as a layer ---
folium.GeoJson(
    data=parques,
    name="Parques",
    style_function=lambda f: {
        "color": "darkgreen",
        "weight": 2,
        "fillColor": "darkgreen",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)

folium.GeoJson(
    data=pracas,
    name="Praças",
    style_function=lambda f: {
        "color": "green",
        "weight": 2,
        "fillColor": "green",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)

# --- layer control + save ---
folium.LayerControl().add_to(m)

In [67]:
m.save("map_green_ju.html")

In [ ]:
parques["type"] = parques["name"].str.split().str[0]

In [60]:
import re

# garantir que a coluna seja string
parques["type"] = parques["type"].astype(str)

# 1. Remover aspas tipográficas antes de "Praça" ou "PRAÇA"
parques["type"] = parques["type"].str.replace(r'^[“”]\s*(?i:praça)', "Praça", regex=True)

# 2. Substituir "PRAÇA" no início por "Praça"
parques["type"] = parques["type"].str.replace(r'^PRAÇA', "Praça", regex=True)

In [ ]:
parques["type"] = parques["type"].str.replace('Park', "Parque", regex=True)

In [62]:
parques["type"] = parques["type"].str.replace('"Praça', "Praça", regex=True)

In [55]:
excluir = ["None", "Renato", "Jardim"]

# manter apenas os que NÃO estão na lista
parques = parques[~parques["type"].isin(excluir)].copy()

In [68]:
mapa = {"Parque": "Parque", "Praça": "Praça"}
parques["category"] = parques["type"].map(mapa).fillna("Outros")

In [72]:
import os
import re
import geopandas as gpd

# gdf base (ex.: 'parques') já deve ter a coluna 'category' criada antes
# valores esperados: "Parque", "Praça", "Outros"
assert "category" in parques.columns, "Crie a coluna 'category' antes de exportar."

OUTDIR = "./data/green_jundiai/by_category"
os.makedirs(OUTDIR, exist_ok=True)

def slug(s: str) -> str:
    """Nome de arquivo seguro (sem espaços/acentos problemáticos)."""
    s = str(s).strip()
    s = re.sub(r"[^\w\-]+", "_", s, flags=re.UNICODE)
    s = re.sub(r"_+", "_", s)
    return s.strip("_") or "categoria"

# (opcional) garanta CRS definido
if parques.crs is None:
    parques = parques.set_crs(4326)

# exportar um .shp por categoria
for cat, gdf_cat in parques.groupby("category"):
    if gdf_cat.empty:
        continue
    fname = f"{slug(cat)}.shp"         # ex.: Parque.shp, Praça.shp, Outros.shp
    outpath = os.path.join(OUTDIR, fname)
    gdf_cat.to_file(outpath, driver="ESRI Shapefile", encoding="utf-8")
    print(f"✅ {cat}: {len(gdf_cat)} feições → {outpath}")

✅ Outros: 16 feições → ./data/green_jundiai/by_category\Outros.shp
✅ Parque: 15 feições → ./data/green_jundiai/by_category\Parque.shp
✅ Praça: 483 feições → ./data/green_jundiai/by_category\Praça.shp


In [1]:
import geopandas as gpd

In [2]:
parque = gpd.read_file("./data/green_jundiai/by_category\Parque.shp").to_crs(4326)
pracas = gpd.read_file("./data/green_jundiai/by_category\Praça.shp").to_crs(4326)
outros = gpd.read_file("./data/green_jundiai/by_category\Outros.shp").to_crs(4326)

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\O'
<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\O'
d:\Users\ivan.cavalcanti\AppData\Local\Temp\6\ipykernel_68012\2870510891.py:1: SyntaxWarning: invalid escape sequence '\P'
  parque = gpd.read_file("./data/green_jundiai/by_category\Parque.shp").to_crs(4326)
d:\Users\ivan.cavalcanti\AppData\Local\Temp\6\ipykernel_68012\2870510891.py:2: SyntaxWarning: invalid escape sequence '\P'
  pracas = gpd.read_file("./data/green_jundiai/by_category\Praça.shp").to_crs(4326)
d:\Users\ivan.cavalcanti\AppData\Local\Temp\6\ipykernel_68012\2870510891.py:3: SyntaxWarning: invalid escape sequence '\O'
  outros = gpd.read_file("./data/green_jundiai/by_category\Outros.shp").to_crs(4326)


In [4]:
parque = gpd.read_file("./data/green_jundiai/parques_jundiai.shp").to_crs(4326)
pracas = gpd.read_file("./data/green_jundiai/pracas_jundiai.shp").to_crs(4326)


In [6]:
pracas

,place,highway,lit,name,surface,addr_city,addr_house,addr_postc,addr_stree,addr_subur,...,amenity,operator,operator_w,police,man_made,waterway,aeroway,athletics,leisure,geometry
0,square,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((-46.84374 -23.192, -46.84374 -23.192..."
1,square,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((-46.84388 -23.19173, -46.84383 -23.1..."


In [ ]:
import geopandas as gpd
import folium


# reproject to WGS84 (lat/lon) if needed
if parque.crs and parque.crs.to_string() != "EPSG:4326":
   parque = parque.to_crs(epsg=4326)

   # reproject to WGS84 (lat/lon) if needed
if pracas.crs and pracas.crs.to_string() != "EPSG:4326":
   pracas = pracas.to_crs(epsg=4326)

      # reproject to WGS84 (lat/lon) if needed
if outros.crs and outros.crs.to_string() != "EPSG:4326":
   outros = outros.to_crs(epsg=4326)

# --- base map ---
m = folium.Map(location=[-15, -55], zoom_start=5, tiles="CartoDB positron")
# --- add shapefile as a layer ---
folium.GeoJson(
    data=parque,
    name="Parque",
    style_function=lambda f: {
        "color": "darkgreen",
        "weight": 2,
        "fillColor": "darkgreen",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)

folium.GeoJson(
    data=pracas,
    name="Praças",
    style_function=lambda f: {
        "color": "lightgreen",
        "weight": 2,
        "fillColor": "lightgreen",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)

folium.GeoJson(
    data=outros,
    name="Praças",
    style_function=lambda f: {
        "color": "blue",
        "weight": 2,
        "fillColor": "lightblue",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)


# --- layer control + save ---
folium.LayerControl().add_to(m)

In [75]:
m.save("map_green_ju_category.html")